<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 8: Deep Learning Libraries

#### Tim Moroney, 2026


A lesson where we build our MNIST classifier using two different deep learning libraries.

# Introduction

So far we thoroughly understand the mathematics behind our MNIST classifier, and have got down to the level of implementing it from first principles.

In this lesson we will take a look at how to implement our model using two different deep learning libraries.  You will see that all the features we have learned can be found here.  We'll also learn about the gradient-based approaches that are used to train the model.

The two libraries we will examine are:

* [Lux](https://lux.csail.mit.edu/stable/): a Julia deep learning library
* [Torch](https://docs.pytorch.org/docs/stable/index.html): a C++ deep learning library that also offers a Python interface

These two examples provide us a good opportunity to examine two different language conventions -- Julia and Python -- and also two different conventions for structuring deep learning libraries -- stateless networks with explicit passing of parameters versus stateful networks with implicit parameter handling. This will all be explained as we go, so don't worry if these concepts don't sound familiar right now.


# Package management
There is no difficulty in calling Python code from Julia, or vice versa, so we shall only need a single notebook environment to experiment with both.  Throughout the notebook we'll generally have a Julia section followed by the analogous Python section.

## Julia

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1mT9XFadzdfK8CWb5a7BYLUkd2RTi2eZc`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "Lux", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "ToeplitzMatrices", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Lux
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentVector
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

# Use existing installation of Python packages
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"
using PythonCall

# Set the random seed for reproducibility
rng = Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

## Python

In [ ]:
np = pyimport("numpy")     # needed for efficient Julia -> Python array conversion
torch = pyimport("torch")  # the Torch deep learning library
nn = torch.nn

# In Python also!
torch.manual_seed(0)

# MNIST dataset

We will load the MNIST dataset, familiar from last week, and split off the first 50,000 images for training.

In [ ]:
# Load the MNIST datset
url = "https://github.com/moroneyt/MXB301/raw/main/resources/MNIST.jld2"
MNISTfile = jldopen(download(url))

images = MNISTfile["images"] / 255f0;  # convert to 0..1 range
labels = MNISTfile["labels"]

N = 50_000
train_images = images[:,:,:,1:N]
train_labels = labels[1:N];

# Single precision vs double precision

In this lesson, we are using the `Float32` data type to represent images. This type, as the name suggests, uses 32 bits rather than the 64 bits of the default `Float64` type.  (These types correspond to `single` and `double` in MATLAB respectively.)  Most deep learning libraries default to `Float32`, which allows for faster computations using less memory, with no perceivable degradation in accuracy for the kinds of tasks to which they are usually put.

In [ ]:
summary(images)

# Training data
We now prepare our training data in the format expected by the two deep learning frameworks.

## Julia
We call our training data $x$ and their corresponding labels, in one-hot encoding, are $y$.  Batches of same are $X$ and $Y$ respectively.

In [ ]:
X = train_images
Y = onehotbatch(train_labels, 0:9)

## Python
Here we need to take care with the conventions. Python uses [row major convention](https://en.wikipedia.org/wiki/Row-_and_column-major_order) for arrays, rather than column major convention used by Julia (also MATLAB and R).  As a result, the most efficient way to process data in Python is to reverse the order of the dimensions compared to Julia. That's OK, we can just reverse the dimensions of $X$ before we pass it through to the Python side.

Let's make a little helper function to do the trick. Notice there is an intermediate conversion to `np.arary` ("numpy array").  This is the _de facto_ n-dimensional numerical array type in Python because its in-built array type is so limited. As a result, torch's `tensor` type supports construction from an `np.array`, hence the conversion goes Julia -> numpy -> torch.

Torch also uses raw indices rather than one-hot encodings for the labels.

In [ ]:
to_torch(X) = torch.tensor(np.array(permutedims(X, ndims(X):-1:1)))
Xp = to_torch(train_images) # suffix p for Python
Yp = torch.tensor(train_labels)

# Hyperparameters

The model hyperparameters are unchanged from last lesson.

In [ ]:
num_classes = 10                                        # digits 0--9
input_size = (28,28)                                    # MNIST image size
in_channels = 1                                         # greyscale input
kernel_size = (5,5)                                     # size of conv filter
feature_map_size = input_size .- kernel_size .+ 1       # size after convolution
out_channels = 8                                        # number of feature maps to generate
flattened_size = prod(feature_map_size) * out_channels  # input size to dense layer

# Network architecture

To build the network architecture, most deep learning libraries provide some kind of "chain" operation for building networks by chaining together basic building blocks.  Our network is pretty simple: we just want a conv layer, flattening and a dense layer.

## Julia

Here is the network built using Lux.  You see we can provide the activation function tanh directly to the conv layer.

In [ ]:
luxnet = Chain(
  Conv(kernel_size, in_channels => out_channels, tanh),
  FlattenLayer(),
  Dense(flattened_size => num_classes)
)

## Python

In Torch it looks much the same, except that Torch doesn't support activation functions in the layers, so you need to put them in yourself as extra steps in the chain.  The naming of various bits and pieces can vary too, e.g. `Linear` instead of `Dense` and the order of the arguments is different.  You say tomato, I say tomato.

In [ ]:
torchnet = nn.Sequential(
    nn.Conv2d(; in_channels, out_channels, kernel_size),
    nn.Tanh(),
    nn.Flatten(),
    nn.Linear(flattened_size, num_classes)
)

# Parameters

When it comes to the network parameters, there is a significant difference in the approach of the two libraries.  In Lux, the parameters are handled _explicitly_.  There will be a variable, called `p`, which is the collection of all the model parameters (which are themselves vectors, matrices, tensors).  This is the approach we are used to.  This explicit approach is also preferred by other modern deep learning libraries you may have heard of, such as TensorFlow and JAX.

In contrast, in Torch there is no explicit parameter array.  Instead, the parameters are stored _implicitly_ as part of the network itself.  We will see the two approaches in action now.

## Julia

To initialise the network parameters we just need to `setup` the network.  This takes care of initialising all the parameter weights and biases using sensible defaults (e.g. Glorot as we covered previously).

In [ ]:
p0, st0 = Lux.setup(rng, luxnet)  # parameters and states are explicit

##
You can see the `p0` variable with all the parameters as we are used to, albeit with names like `p0.layer_1.weight` rather than `p0.F`.

In [ ]:
p0.layer_1.weight  # we just called this F last week

##
There is also an `st` variable, which represents the _state_ of the network. Some networks, albeit not this lesson's, have additional state that gets updated during training.  We will see examples in later lessons.  But for now, we can ignore this state variable, since it's just empty.

In [ ]:
st0  # our network has no state

##
We evaluate the neural net by passing in the data, the parameters and state (the latter being empty). The network output is returned, along with the updated state.

In [ ]:
predictions, st = luxnet(X, p0, st0)  # parameters and state passed explicitly

## Python

Torch is the prime example of the older, implicit style of deep learning frameworks.  In this style, the model parameters, their gradients, and any network state all live within the model -- there are no parameter variables or gradients passed as inputs or returned as outputs. This implicit style has gone out of fashion, but it's still good to get a taste of both approaches.

The parameters in our Torch network have in fact already been initialised.  So there's nothing else to do here, unless we want to take a look at them for fun.

In [ ]:
# we have to extract the parameters from the network if we want to view them
torchparams = pylist(torchnet.parameters())

# this is F
torchparams[0]

##
To evaluate the network in Torch, we only pass the data as input.

In [ ]:
torchnet(Xp)  # parameters and state are implicit

# Loss function

You may have been wondering why we didn't include softmax in the final step of the network (I hope you were!).  The answer is that it's very common that a softmax output gets combined with a cross entropy loss.  That's certainly been the case with our models.  And in that case, it's more efficient, and potentially more reliable numerically, to roll the softmax and cross entropy calculations into one step.

This is quite easy to see actually.  Recall the definition of the two operations:

softmax:
$$
y_i = \frac{\textrm{e}^{z_i}}{\sum_j \textrm{e}^{z_j}}
$$

cross-entropy:
$$
\textrm{crossentropy}(y, \hat{y}) = -\log (\hat{y}_k)\qquad (y\textrm{ is one-hot at index }k)
$$

It's kind of pointless taking the exponential in the numerator of softmax, only to undo it by taking the log in the cross-entropy.  So in combination, we have simply

softmax + cross-entropy loss:
$$
L = \log \sum_j \textrm{e}^{z_j} - z_k
$$



In fact we already used this shortcut to derive the simplified formula for the _gradient_ of softmax + cross-entropy in the backward pass.  We're only now also using this simplification in the forward pass.  (This formulation also allows more robust implementations that avoid over- and under-flow in the calculations of the exponentials $\textrm{e}^{z_j}$, which deep learning frameworks make sure to get right.)

Recall the terminology that the inputs to softmax are referred to as **logits**.  So as of this week, the raw outputs of our network are logits.  Previously we would have fed these logits into softmax.  But now, we will feed them directly into the combined softmax + cross-entropy loss, relying on the deep learning library to merge the two calculations internally.

##Julia

We just need to pass `logits = true` to the `CrossEntropyLoss` to indicate that the inputs are indeed logits.

In [ ]:
luxloss = CrossEntropyLoss(logits = true)

## Python

Logits as input is already the default in Torch.

In [ ]:
torchloss = nn.CrossEntropyLoss()  # logits as input is the default

# Training overview
When it comes to training in deep learning libraries, gradient descent, or one of its variants, dominate.  In whichever framework you're using, the training loop is typically written in a form resembling the following:

```
for epoch = 1:num_epochs
  for (X, Y) in data_loader
    Compute loss (evaluate the forward pass)
    Compute gradient (evaluate the backward pass using reverse mode AD)
    Take a gradient step
  end
end
```
One full iteration of the outer loop is called an **epoch**.

The `data_loader` is a convenience provided by the deep learning library to give you one batch of data at a time.  Notice that we are taking a gradient step _each and every time_ we process a batch.  We are _not_ waiting to accumulate the whole gradient before taking a step.

This is a significant difference in perspective to our previous approach using L-BFGS, Newton, and friends.  There, we made sure to accumulate the full gradient
$$
\nabla_p L(p) = \sum_{j=1}^B \nabla_p \ell(y^{(j)}, M_p(x^{(j)})) + \sum_{j=B+1}^{2B} \nabla_p \ell(y^{(j)}, M_p(x^{(j)})) + \ldots + \sum_{j={N}-B+1}^{N} \nabla_p \ell(y^{(j)}, M_p(x^{(j)})) \,,
$$

where $B$ is the batch size and $\ell$ is the per-sample loss function.  One step per epoch, based on accumulating all the terms.

Whereas for gradient descent-based methods, it's quite usual to take a step as soon as you have calculated a gradient approximation using a _single batch_, for example
$$
\nabla_p L(p) \approx \sum_{j=1}^B \nabla_p \ell(y^{(j)}, M_p(x^{(j)}))\,.
$$

Finally, if the `data_loader` _randomises_ the batches with each epoch, then we may not even be optimising the exact same objective function with each epoch.  Instead, we are introducing _stochasticity_ into the objective, making this a so-called **stochastic gradient descent** method.





# Data Loader
Both frameworks provide a type called `DataLoader` that handles the batching of the data (randomised, if `shuffle = true`) for each epoch.

## Julia

In [ ]:
batchsize = 100
lux_loader = DataLoader((X, Y); batchsize, shuffle=true)

## Python

In [ ]:
torch_dataset = torch.utils.data.TensorDataset(Xp, Yp)
torch_loader = torch.utils.data.DataLoader(torch_dataset, batchsize, shuffle=true)

# Stochastic gradient descent

Permitting stochasticity in the objective function is a nice generalisation, which we will put to even more use in the next lesson.  It does introduce a new complication to the optimisation process that we have to take seriously though: we no longer have access to the full gradient $\nabla L$.  Instead at any step $k$ of the iteration we have only the random estimate
$$
g^{(k)} = \nabla L_k(p^{(k)}) \approx \nabla L(p^{(k)})
$$
where we use the notation $L_k$ to denote the loss function computed over the $k$th batch only.

The index $k$ here is _not_ counting epochs.  With each step taken for a single batch, $k$ increases by one.  So after one full epoch, $k = \lceil N/B\rceil$ (ceiling because $B$ may not divide $N$ evenly).  Then we start the next epoch, returning to the first batch to process them all again one by one, and $k$ continues to increase.

The correct way to treat this mathematically is to view each $g^{(k)}$ as a _random sample_ with the property
$$
\mathbb{E}[g^{(k)}] = \nabla L(p^{(k)})\,.
$$

## Averaging / momentum

Because the sampled gradient direction $g^{(k)}$ is _noisy_, it's particularly dubious to simply use the vanilla gradient descent update

$$p^{(k+1)} = p^{(k)} - \alpha\, g^{(k)}\,.$$

Instead, a standard idea in stochastic gradient descent is to use a weighted combination of the previous search direction, which we will denote by $m^{(k-1)}$, and the newly-sampled noisy gradient estimate:

$$
\begin{align*}
m^{(k)} &= \beta_1 m^{(k-1)} + (1 - \beta_1) g^{(k)}\\
p^{(k+1)} &= p^{(k)} - \alpha\, m^{(k)}\,.
\end{align*}
$$

In fact this idea of using a linear combination of the previous search direction and the new gradient is already familiar to us from our study of the conjugate gradient method in lesson 5.  There, it fell out of the mathematics directly.  Here, we are simply imposing this same idea as a way to deal with the random fluctuations associated with the sampled gradient vectors in stochastic gradient descent.

Expanding this recursion, assuming the initial iterate is defined as $m^{(0)} = 0$ (more on that later), we find

$$
m^{(k)} = (1 - \beta_1) \sum_{j=1}^k {\beta_1}^{k-j}\, g^{(j)}
$$

so that the search direction is a geometric average of _all_ previous gradients, with the more recent ones weighted more heavily.  This makes sense: when randomness is part of the picture, it's usually more robust to use an averaged quantity rather than a raw, noisy one.  This **first moment estimate** is just that.

## Component-wise learning rate

A further improvement can be made by keeping track of the weighted _second_ moment of the gradient.  Call this running average $v^{(k)}$, defined analogously by
$$
v^{(k)} = \beta_2 v^{(k-1)} + (1 - \beta_2) {g^{(k)}}^{\circ 2}
$$
(where we recall $g^{\circ 2}$ means component-wise squaring).

Expanding the recursion again,
$$
v^{(k)} = (1 - \beta_2) \sum_{j=1}^k {\beta_2}^{k-j}\, {g^{(j)}}^{\circ 2}\,.
$$

What is this averaged quantity useful for?  It measures the component-wise mean square gradient -- large values imply the corresponding component in the gradient has consistently large magnitude.  We should be wary of imposing a large learning rate on such a component, since it implies a very large step in the corresponding parameter, potentially beyond the validity of the descent direction.  Conversely, a very small value of the mean square gradient suggests a larger value of the learning rate would be suitable for that component.

Hence we are drawn to propose a gradient descent update that uses a _per-component learning rate_, dictated by the values of the mean square gradient.  Here it is component-wise:

$$
p_i^{(k+1)} = p_i^{(k)} - \alpha\, \frac{m_i^{(k)}}{{\sqrt{v_i^{(k)}}} + \epsilon}
$$

and here in terms of vector operations:

$$
p^{(k+1)} = p^{(k)} - \alpha\, m^{(k)} \oslash ({v^{(k)}}^{\circ \frac{1}{2}} + \epsilon)
$$

where $\oslash$ denotes element-wise division, and $\epsilon$ is a vector of small values (e.g. $10^{-8}$) to avoid division by zero.

Equivalently, if we let $D$ denote the diagonal matrix
$$
D = \textrm{diag}({v^{(k)}}^{\circ \frac{1}{2}} + \epsilon)
$$
we may write
$$
p^{(k+1)} = p^{(k)} - \alpha\, D^{-1} m^{(k)}
$$

which has the flavour of a Newton-style update with a (very crude) _diagonal_ approximation of the true Hessian matrix $H$.  Now don't take this comparison too seriously -- this is still very much a first-order method.  No actual curvature information is being used.  Indeed if you analyse the dimensions of this update (exercises!) you can see it still doesn't make sense dimensionally.  So more correctly you could describe this as a "diagonally preconditioned gradient descent" or similar.  But there's no denying that component-wise learning rates are a step in the right direction (pun intended!) for improving the performance of stochastic gradient descent.

## Bias correction

The method we have derived is essentially the Adaptive Moment Estimation method, or [Adam](https://arxiv.org/pdf/1412.6980).  It's a popular method for stochastic gradient descent, and is the method we shall apply in this lesson.  However, it is not quite ready to roll in its current form.

The problem is the adaptive moment estimates
$$
\begin{align*}
m^{(k)} &= \beta_1 m^{(k-1)} + (1 - \beta_1) g^{(k)}\\
v^{(k)} &= \beta_2 v^{(k-1)} + (1 - \beta_2) {g^{(k)}}^{\circ 2}
\end{align*}
$$

need a starting point, which are taken to be $m^{(0)} = 0$ and $v^{(0)} = 0$ respectively.  This introduces a _bias_ in the moment estimates towards zero in the early iterations.

We can analyse this effect for the first moment, assuming the stochastic gradients have constant mean
$$
\mathbb{E}[g^{(k)}] = \mu.
$$

Then
$$
\begin{align*}
\mathbb{E}[m^{(k)}] &= \mathbb{E}\left[(1 - \beta_1) \sum_{j=1}^k {\beta_1}^{k-j}\, g^{(j)}\right] \\
&= (1 - \beta_1) \sum_{j=1}^k {\beta_1}^{k-j}\, \mathbb{E}[g^{(j)}] \\
&= (1 - \beta_1) \sum_{j=1}^k {\beta_1}^{k-j}\, \mu \\
&= (1 - \beta_1) \frac{1 - {\beta_1}^k}{1 - \beta_1} \mu \\
&= (1 - {\beta_1}^k) \mu\,.
\end{align*}
$$

Similarly, if $\mathbb{E}[{g^{(k)}}^{\circ 2}] = \nu$ then (exercises!)
$$
\mathbb{E}[v^{(k)}] = (1 - {\beta_2}^k) \nu\,.
$$




So the Adam update uses the **bias-corrected moment estimates**
$$
\hat{m}^{(k)} = \frac{m^{(k)}}{1 - {\beta_1}^k}\qquad\textrm{and}\qquad \hat{v}^{(k)} = \frac{v^{(k)}}{1 - {\beta_2}^k}\,.
$$

Substituting these into the update formulas, we derive the full Adam step:
$$
g^{(k)} = \nabla_p\, L_k(p^{(k)})
$$

$$
m^{(k)} = \beta_1 m^{(k-1)} + (1 - \beta_1) g^{(k)}
$$

$$
v^{(k)} = \beta_2 v^{(k-1)} + (1 - \beta_2) {g^{(k)}}^{\circ 2}
$$

$$
\hat{m}^{(k)} = \frac{m^{(k)}}{1 - {\beta_1}^k},\qquad \hat{v}^{(k)} = \frac{v^{(k)}}{1 - {\beta_2}^k}
$$

$$
p^{(k+1)} = p^{(k)} - \alpha\, \hat{m}^{(k)} \oslash ({\hat{v}^{(k)}}^{\circ \frac{1}{2}} + \epsilon)\,.
$$



## Julia

In Lux, you first choose an optimiser.  We'll use Adam with all the default parameter values (which are the standard ones recommended in the paper).

In [ ]:
optimiser = Lux.Optimisers.Adam()

##
Then you build a `TrainState` object to keep track of the iteration history, and pass it the parameters, state, and the optimiser.

In [ ]:
trainer = Training.TrainState(luxnet, p0, st0, optimiser) # keeps track of the training history

## Python

In Torch, the `Adam` optimiser needs to be linked to the model parameters.  This is how it knows which parameters to optimise.

Notice that here the "implicit" style of Torch is momentarily abandoned, because now you kind-of-sort-of do need to pass the parameters as an input for this one step.

In [ ]:
torchopt = torch.optim.Adam(torchnet.parameters())

# Training loop

For the training loop there is a little more daylight between the two frameworks, owing again to the explicit vs implicit handling of parameters and gradients.

## Julia
For each batch, you compute the gradients using reverse mode autodiff, then apply them to perform a step.  Since we're using Adam, performing a step involves carrying out all the calculations listed above for Adam, including the bookkeeping to update the moment estimates, adapting the learning rate, and of course actually taking the gradient step to update the parameters.

In [ ]:
autodiff = AutoZygote()  # reverse mode AD

for (X,Y) in lux_loader

    # Compute loss and gradient: lossval = Lₖ(p;X,Y), g = ∇ₚ Lₖ(p;X,Y)
    g, lossval, extras, trainer = Training.compute_gradients(autodiff, luxloss, (X, Y), trainer)

    # Take gradient step and perform associated bookkeeping for the optimiser
    trainer = Training.apply_gradients(trainer, g)
end

## Python

And here is one step of training in Torch.  Because it handles gradients implicitly, you have to remember to zero out the gradient beforehand each time.

In [ ]:
for (X,Y) in torch_loader
    torchopt.zero_grad()    # necessary because of the implicit handling of gradients

    # Compute loss
    lossval = torchloss(torchnet(X), Y)

    # Compute gradient
    lossval.backward()

    # Take gradient step
    torchopt.step()
end

# Full training loop

Here we go at last, the full training loop.  Let's do 30 epochs.

We'll use `@time` to time how long it takes for the entire training process and `@show` to keep an eye on the progress with each epoch.  Expect it to take around 3 minutes.



In [ ]:
num_epochs = 30

# Julia



In [ ]:
@time for epoch = 1:num_epochs
    total_loss = 0.0f0
    for (X,Y) in lux_loader
        g, lossval, extras, trainer = Training.compute_gradients(autodiff, luxloss, (X, Y), trainer)
        trainer = Training.apply_gradients(trainer, g)
        total_loss += lossval * batchsize  # CrossEntropyLoss computes mean over batchsize
    end
    avg_loss = total_loss / N
    @show (epoch, avg_loss)
end

# explicit parameter handling
p = trainer.parameters;

# Python

In [ ]:
@time for epoch = 1:num_epochs
    total_loss = 0.0f0
    for (X,Y) in torch_loader
        torchopt.zero_grad()
        Ŷ = torchnet(X)
        lossval = torchloss(Ŷ, Y)
        lossval.backward()
        torchopt.step()
        total_loss += lossval.item() * batchsize   # CrossEntropyLoss computes mean over batchsize
    end
    avg_loss = total_loss / N
    @show (epoch, avg_loss)
    epoch % 10 == 0 && GC.gc() # frees the memory that Python would otherwise hold on to
end

# Inference
Let's now test our trained models on a test set of images.  We'll use the last 10,000 images in the set.

In [ ]:
test_images = images[:,:,:,end-9999:end]
test_labels = labels[end-9999:end];

# Julia
We want the one-hot representation of the labels.

In [ ]:
Xtest = test_images
Ytest = onehotbatch(test_labels, 0:9)

# Python

In [ ]:
Xtestp = to_torch(test_images)
Ytestp = to_torch(test_labels)

# Model function

We still need to define our convenience `model` function, and now it's not just a wrapper around the forward pass.  It has to include the softmax step at the end so that we do actually get probabilities out when we perform inference.

## Julia

In [ ]:
# pass in data, parameters, state
luxmodel(X) = softmax(luxnet(X, p, st)[1])

# try it out on our test set
Ŷtest = luxmodel(Xtest)

## Python

In [ ]:
# pass in data only since parameters and state are implicit
torchmodel(X) = nn.functional.softmax(torchnet(X); dim=1)

# try it out on our test set
Ŷtestp = torchmodel(Xtestp)

# Model classifications
From the probabilities output from the model, we can take the argmax to find the model's predicted labels.

## Julia

In [ ]:
# Model predictions
lux_predictions = argmax.(eachcol(Ŷtest)) .- 1

# Find which images were misclassified
lux_misclassified = findall(lux_predictions .!= test_labels)

## Python

In [ ]:
# Model predictions
torch_predictions = pyconvert(Vector, torch.argmax(Ŷtestp; dim=1))

# Find which images were misclassified
torch_misclassified = findall(torch_predictions .!= test_labels)

# Comparison
It's interesting to compare the results of the two networks on the test set.  You can see that they are are confused on many of the same images.

Both models are using the same network architecture, so any differences are just an artefact of the randomised initial parameter values, plus whatever subtle differences there may be in the implementation of the various components in the model and optimiser.

In [ ]:
println(
  "Lux   number of misclassified test images: ", length(lux_misclassified), " out of ", size(Ytest,2),
  ". Accuracy: ", 1 - length(lux_misclassified)/size(Ytest,2))
println(
  "Torch number of misclassified test images: ", length(torch_misclassified), " out of ", size(Ytest,2),
  ". Accuracy: ", 1 - length(torch_misclassified)/size(Ytest,2))

both_misclassified = intersect(lux_misclassified, torch_misclassified)
println("Both misclassified: ", length(both_misclassified), " images.")

# Overtraining
You might think that training for even more epochs will just improve the performance.  But this is not so.  It may improve the performance on the _training set_, but there is no guarantee it will improve performance on the _previously unseeen_ test set -- in fact it can result in _worse_ performance on the test set, not better.  This phenomenon is called **overtraining**. The reduction in performance comes from the model learning to exploit specific idiosyncrasies of the training data in order to reduce the total loss.  While this will help it to improve its prediction accuracy on the training set, if the features it has learned to exploit are not generalisable to data it hasn't seen, then it will perform poorly on any new data that it wasn't trained on.

If we were taking this more seriously, we might even test the model after each epoch on new images it hadn't seen, and stop the training process once the performance on the new data stalled, or started deteriorating.

# Conclusion

In this lesson we learned:

* how to build an MNIST image classifier using high-level deep learning libraries rather than coding layers manually

* how training data and labels are loaded, preprocessed, and arranged in the tensor shapes expected by different frameworks

* about the Adam optimiser for stochastic gradient descent

* the standard form of the training loop in different frameworks

* how library design differs between frameworks particularly in how parameters are managed

In the next lesson we will learn about a new kind of model, which also has important applications in image processing, called a convolutional variational autoencoder.